# Limpeza e Push — Code49 (Caixeta / IMOBPARC / MK Prime)

Pipeline de limpeza dos dados Code49 ate push via API.

In [ ]:
import datawork
datawork.setup()

In [ ]:
from datawork.io.loaders import load_scraping_results

FONTE_ID = ""  # Preencher com UUID da fonte Code49
df_raw = load_scraping_results(FONTE_ID)
print(f"{len(df_raw)} registros crus")

In [ ]:
from datawork.pipeline import PipelineRunner
from datawork.pipeline.stages import (
    drop_empty_rows, normalize_areas, geocode_addresses,
    compute_dedup_hash, drop_duplicates_by_hash,
    generate_titulo, rename_to_silver, select_columns,
)
from datawork.pipeline.stages_scraping import (
    normalize_scraped_titles, normalize_price_scraped,
    detect_tipo_from_title, extract_features_from_description,
    fill_source_metadata,
)
from datawork.pipeline.stages_code49 import flatten_code49_response

pipeline = (
    PipelineRunner("code49_clean")
    .add("flatten", flatten_code49_response)
    .add("empty", drop_empty_rows)
    .add("titles", normalize_scraped_titles)
    .add("tipo", detect_tipo_from_title)
    .add("areas", normalize_areas)
    .add("prices", normalize_price_scraped)
    .add("features", extract_features_from_description)
    .add("geocode", geocode_addresses)
    .add("metadata", lambda df: fill_source_metadata(df, "Code49", "code49.com.br"))
    .add("titulo", generate_titulo)
    .add("dedup", compute_dedup_hash)
    .add("dedup_drop", drop_duplicates_by_hash)
    .add("rename", rename_to_silver)
    .add("select", select_columns)
)

df_clean = pipeline.run(df_raw)
pipeline.summary()

In [ ]:
from datawork.display import show_sample, show_stats
from datawork.profiling import completeness_report

show_sample(df_clean, n=5)
show_stats(df_clean)
completeness_report(df_clean)

In [ ]:
from datawork.contracts.silver import CleanRecordSchema

CleanRecordSchema.validate(df_clean, lazy=True)
print("Schema validation PASSED")

In [ ]:
# Push para API (descomentar quando pronto)
# from datawork.io.pushers import push_clean_to_api
# result = push_clean_to_api(df_clean, FONTE_ID)
# print(result)